# Smart Fire Detection - YOLOv8 training on Google Colab
Owner: **Tanishka Yadav**. Runtime → Change runtime type → **T4 GPU** before running.

Steps: mount Drive → get dataset → train → evaluate → download `best.pt` into `model/weights/` of the project.

In [ ]:
!nvidia-smi
!pip install -q ultralytics roboflow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# keep everything on Drive so a disconnect doesn't lose the training run
PROJECT = '/content/drive/MyDrive/smart-fire-detection'
import os; os.makedirs(PROJECT, exist_ok=True)

## Option A - public dataset from Roboflow Universe
Search Roboflow Universe for a fire + smoke dataset (e.g. *D-Fire* style datasets), click **Download → YOLOv8**, and paste the snippet here. Put your API key in Colab *Secrets* as `ROBOFLOW_API_KEY`, never in the notebook.

In [ ]:
from google.colab import userdata
from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
# replace with the workspace / project / version shown on Roboflow
project = rf.workspace('WORKSPACE').project('PROJECT')
dataset = project.version(1).download('yolov8', location=f'{PROJECT}/downloads/public')

## Option B - your own labelled images
Upload the YOLO-format export of your self-collected images to `MyDrive/smart-fire-detection/labelled/` and the project code to `MyDrive/smart-fire-detection/code/`, then merge + split with `prepare_dataset.py`:

In [ ]:
%cd {PROJECT}/code
!python -m model.prepare_dataset --inputs {PROJECT}/downloads/public {PROJECT}/labelled --out {PROJECT}/datasets/fire --classes fire smoke --class-map "Fire=fire" "Smoke=smoke"

## Train

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')   # try yolov8s.pt too and compare in the report
results = model.train(data=f'{PROJECT}/datasets/fire/data.yaml', epochs=100, imgsz=640, batch=32,
                      patience=25, project=f'{PROJECT}/runs', name='yolov8n_fire', exist_ok=True,
                      hsv_h=0.010, hsv_s=0.6, hsv_v=0.4, mixup=0.1, degrees=5.0)

In [ ]:
# resume after a Colab disconnect
# model = YOLO(f'{PROJECT}/runs/yolov8n_fire/weights/last.pt'); model.train(resume=True)

## Evaluate on the test split

In [ ]:
best = YOLO(f'{PROJECT}/runs/yolov8n_fire/weights/best.pt')
m = best.val(data=f'{PROJECT}/datasets/fire/data.yaml', split='test')
print('mAP50', m.box.map50, 'mAP50-95', m.box.map, 'P', m.box.mp, 'R', m.box.mr)

In [ ]:
from IPython.display import Image, display
for f in ['results.png', 'confusion_matrix.png', 'PR_curve.png']:
    display(Image(f'{PROJECT}/runs/yolov8n_fire/{f}', width=800))

## Download weights
Copy `best.pt` to `model/weights/best.pt` in the project on your laptop. Also export NCNN for the Raspberry Pi:

In [ ]:
best.export(format='ncnn', imgsz=320)
from google.colab import files
files.download(f'{PROJECT}/runs/yolov8n_fire/weights/best.pt')